## Reformat Catalog Datasets

In [ ]:
import pandas as pd
import requests
import json

### Messier Catalog

In [ ]:
# reformat messier dataset
df = pd.read_csv('./object_datasets/MessierObjects.csv') # there is an extra row and header is wonky

# change magnitudes
maglist = list(df.Magnitude)
for i in range(len(maglist)):
    try:
        maglist[i] = float(maglist[i])
    except:
        print(i, ":", maglist[i])
maglist[39] = float(9)
df.Magnitude = maglist

def get_max_size(size):
    size = size.replace('Size: ', '').split('x') # remove 'size: ' then split into two dimensions
    return max([float(val) for val in size]) # choose greater of two dimensions if there are two
sizelist = list(df.Info)
for i in range(len(sizelist)):
    try:
        sizelist[i] = get_max_size(sizelist[i])
    except:
        print(i, ":", sizelist[i])
sizelist[39] = float(0.82)
df.Info = sizelist
df.rename(columns = {'Info':'Size'}, inplace = True)

# add M to ObjectNum
nums = list(df.ObjectNum.astype(int))
nums = [f"M{str(num)}" for num in nums]
df.ObjectNum = nums

df.reset_index(drop=True, inplace=True)
df = df.reindex(columns=['ObjectNum', 'Name', 'Type', 'Constellation', 'Magnitude', 'Size', 'RAHour', 'RAMinute', 'DecSign', 'DecDeg', 'DecMinute', 'Distance (ly)'])
df.to_csv('./object_datasets/object_data_messier.csv', index=False)

### NGC Catalog

In [ ]:
# load file
df = pd.read_csv('./object_datasets/NGCObjects.csv')
print(len(df))
# remove NA values from magnitude
df = df[df.Magnitude.notna()]
df.Magnitude.astype(float)

# change name to just NGC name
# not worrying about alternate names that are used
df.Name = "NGC " + df.ObjectNum.astype(str)

# add constellation names
# string with constellations and their abbreviations
const_string = "And	Andromeda Ant	Antlia Aps	Apus Aql	Aquila Aqr	Aquarius Ara	Ara Ari	Aries Aur	Auriga Boo	Boötes Cae	Caelum Cam	Camelopardalis Cap	Capricornus Car	Carina Cas	Cassiopeia Cen	Centaurus Cep	Cepheus Cet	Cetus Cha	Chameleon Cir	Circinus CMa	Canis_Major CMi	Canis_Minor Cnc	Cancer Col	Columba Com	Coma_Berenices CrA	Corona_Australis CrB	Corona_Borealis Crt	Crater Cru	Crux Crv	Corvus CVn	Canes_Venatici Cyg	Cygnus Del	Delphinus Dor	Dorado Dra	Draco Equ	Equuleus Eri	Eridanus For	Fornax Gem	Gemini Gru	Grus Her	Hercules Hor	Horologium Hya	Hydra Hyi	Hydrus Ind	Indus Lac	Lacerta Leo	Leo Lep	Lepus Lib	Libra LMi	Leo_Minor Lup	Lupus Lyn	Lynx Lyr	Lyra Men	Mensa Mic	Microscopium Mon	Monoceros Mus	Musca Nor	Norma Oct	Octans Oph	Ophiuchus Ori	Orion Pav	Pavo Peg	Pegasus Per	Perseus Phe	Phoenix Pic	Pictor PsA	Piscis_Austrinus Psc	Pisces Pup	Puppis Pyx	Pyxis Ret	Reticulum Scl	Sculptor Sco	Scorpius Sct	Scutum Ser	Serpens Sex	Sextans Sge	Sagitta Sgr	Sagittarius Tau	Taurus Tel	Telescopium TrA	Triangulum_Australe Tri	Triangulum Tuc	Tucana UMa	Ursa_Major UMi	Ursa_Minor Vel	Vela Vir	Virgo Vol	Volans Vul	Vulpecula"
dictlist = []
for pair in const_string.split(' '): # splits into key, pair for each constellation
    pair = pair.split('	')
    dictlist.append(pair[0])
    dictlist.append(pair[1])
it_const = iter(dictlist)
abbreviations_dict = dict(zip(it_const, it_const)) # dictionary
df.Constellation = df.Constellation.replace(abbreviations_dict)
print(len(df))
# info section
# just using size from the this section, may use other information later
df = df.loc[df.Info.str.contains("Size: ")] # contains size info
df.Info = df.Info.str.slice(6,9).astype(float) # just get the number for size
df.rename(columns = {'Info':'Size'}, inplace = True) # rename to size

# rearrange columns
df = df.reindex(columns=['ObjectNum', 'Name', 'Type', 'Constellation', 'Magnitude', 'Size', 'RAHour', 'RAMinute', 'DecSign', 'DecDeg', 'DecMinute', 'Distance (ly)'])
df.reset_index(drop=True, inplace=True)

# save to csv
df.to_csv('./object_datasets/object_data_ngc.csv', index=False)

## Combined

In [ ]:
df_messier = pd.read_csv('./object_datasets/object_data_messier.csv')
df_ngc = pd.read_csv('./object_datasets/object_data_ngc.csv')

df_messier['Full_Name'] = df_messier.Name
df_ngc['Full_Name'] = df_ngc.Name
df_messier.Name = df_messier.Full_Name.str.slice(0, 8)

messier_ngc_names = list(df_messier[df_messier.Name.str.slice(0,3) == "NGC"].Name)
for i in range(len(messier_ngc_names)):
    if messier_ngc_names[i][-1] == ' ':
        print(i, ":", messier_ngc_names[i])
for i in [29, 30, 31, 70, 72, 106]:
    df_messier.loc[df_messier.Name == messier_ngc_names[i], ['Name']] = messier_ngc_names[i][:7]

# planets and moon
df_solar_system = pd.DataFrame({'ObjectNum': ['Mercury', 'Venus', 'Mars', 'Jupiter', 'Saturn', 'Uranus', 'Neptune', 'Moon'], 
 'Name': ['Mercury', 'Venus', 'Mars', 'Jupiter', 'Saturn', 'Uranus', 'Neptune', 'Moon'],
 'Type': ['Planet', 'Planet', 'Planet', 'Planet', 'Planet', 'Planet', 'Planet', 'Moon'],
 'Constellation':  ['---', '---', '---', '---', '---', '---', '---', '---'],
 'Magnitude': [None, None, None, None, None, None, None, None], #[None, None, None, None, None, None, None],
 'Size': [None, None, None, None, None, None, None, None], 
'RAHour': ['---', '---', '---', '---', '---', '---', '---', '---'],
'RAMinute': ['---', '---', '---', '---', '---', '---', '---', '---'],
'DecSign': ['---', '---', '---', '---', '---', '---', '---', '---'],
'DecDeg': ['---', '---', '---', '---', '---', '---', '---', '---'],
'DecMinute': ['---', '---', '---', '---', '---', '---', '---', '---'],
'Distance (ly)': [None, None, None, None, None, None, None, None],
'Full_Name': ['Mercury', 'Venus', 'Mars', 'Jupiter', 'Saturn', 'Uranus', 'Neptune', 'Moon']})
df_solar_system.to_csv('./object_datasets/solar_system_data.csv', index=False)  

# change ObjectNum to name for requesting stellarium
df_ngc.loc[:, ['ObjectNum']] = df_ngc.Name

df_new = pd.concat([df_ngc, df_messier, df_solar_system])
df_new = df_new.drop_duplicates(subset=['Name'], keep='last')
df_new['Name'] = df_new.Full_Name
df_new = df_new.drop(columns=['Full_Name'])
df_new.reset_index(drop=True, inplace=True)

In [ ]:
# remove ngcs which aren't able to be accessed by stellarium
# this takes like 2 ish mins
unused_ngcs, distance_list = [], []
for i in range(len(df_new)):
    obj_name = str(df_new.ObjectNum[i]).replace(' ', '')
    try:
        response = requests.get(f"http://localhost:8090/api/objects/info?name={obj_name}&format=json")
        info = json.loads(response.text)
        distance_list.append((i, obj_name))
    except:
        unused_ngcs.append(str(df_new.ObjectNum[i]))

In [ ]:
df_new = df_new[df_new.ObjectNum.isin(unused_ngcs) == False] # drop the unused ones
df_new.reset_index(drop=True, inplace=True)

# save to csv
df_new.to_csv('./object_datasets/object_data_complete.csv', index=False)